In [1]:
!pip install openai python-dotenv


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install groq


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import re
import json
from datetime import datetime
from groq import Groq
import os

In [4]:
# saving groq api 
os.environ["GROQ_API_KEY"] = "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"

In [5]:
# loading groq api
api_key = os.getenv("GROQ_API_KEY")
client = Groq(api_key=api_key)

In [6]:
chat_history = []

In [7]:
# storing meage for chat hitory
def store_message(role, content):
    chat_history.append({
        "role": role,
        "content": content,
        "time": str(datetime.now())
    })

In [8]:
# keyword to help in filter message
emergency_keywords = [
    "chest pain",
    "can't breathe",
    "cannot breathe",
    "unconscious",
    "heavy bleeding",
    "seizure"
]

overdose_keywords = [
    "overdose",
    "too many pills",
    "poison",
    "swallowed bleach"
]

selfharm_keywords = [
    "suicide",
    "kill myself",
    "self harm"
]

In [9]:
def keyword_prefilter(user_input):
    text = user_input.lower()

    for word in emergency_keywords:
        if word in text:
            return "EMERGENCY"

    for word in overdose_keywords:
        if word in text:
            return "OVERDOSE"

    for word in selfharm_keywords:
        if word in text:
            return "SELF_HARM"

    return "SAFE"

In [10]:
# aking AI to categorize user input to help in filter
def ai_prefilter(user_input):
    prompt = f"""
Classify this health query into ONE category only:
SAFE
EMERGENCY
OVERDOSE
SELF_HARM
PRESCRIPTION_RISK

Query: {user_input}
Return only category.
"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content.strip()

In [11]:
def hybrid_prefilter(user_input):
    keyword_result = keyword_prefilter(user_input)

    if keyword_result != "SAFE":
        return keyword_result

    return ai_prefilter(user_input)

In [12]:
# safety message 
def safety_response(category):
    responses = {
        "EMERGENCY": "This may be a medical emergency. Please seek immediate medical attention.",
        "OVERDOSE": "Possible overdose detected. Contact emergency services or poison control immediately.",
        "SELF_HARM": "Please reach out to emergency support or a mental health professional immediately.",
        "PRESCRIPTION_RISK": "I cannot provide exact prescription or dosage advice. Consult a licensed doctor."
    }

    return responses.get(category)

In [13]:
# buiding prompt and storing message 
def build_prompt(user_input):
    system_prompt = """
You are a helpful medical assistant chatbot.

Rules:
1. Provide general health education only.
2. Do NOT diagnose diseases.
3. Do NOT prescribe exact medication dosages.
4. Use simple and friendly language.
5. If symptoms sound serious, advise consulting a doctor.
6. Keep answers concise and clear.
"""
    
    messages = [{"role": "system", "content": system_prompt}]
    
    # Add previous chat history (memory)
    for msg in chat_history:
        messages.append({
            "role": msg["role"],
            "content": msg["content"]
        })
    
    # Add current user message
    messages.append({"role": "user", "content": user_input})
    
    return messages

In [14]:
# getting response from AI by passing prompted message with user input 
def get_llm_response(user_input):
    messages = build_prompt(user_input)

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",   # Groq model
        messages=messages,
        temperature=0.4
    )

    return response.choices[0].message.content

In [15]:
unsafe_patterns = [
    "you definitely have",
    "you certainly have",
    "confirmed diagnosis",
    "take 5 tablets",
    "take 10 pills",
    "every hour"
]

In [16]:
def keyword_output_validator(response):
    text = response.lower()

    for pattern in unsafe_patterns:
        if pattern in text:
            return "UNSAFE"

    return "SAFE"

In [17]:
def ai_output_validator(response):
    prompt = f"""
Check whether this medical response is safe.

Unsafe if:
- Gives definite diagnosis
- Gives dangerous dosage
- Gives harmful advice

Response:
{response}

Return only:
SAFE
or
UNSAFE
"""
    
    validation = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )

    return validation.choices[0].message.content.strip()

In [18]:
# validating output
def hybrid_output_validator(response):
    keyword_result = keyword_output_validator(response)

    if keyword_result == "UNSAFE":
        return "UNSAFE"

    return ai_output_validator(response)

In [19]:
def fallback_response():
    return """
I cannot provide diagnosis or specific medication instructions.
Please consult a licensed healthcare professional for proper medical advice.
"""

In [20]:
# complete code
def health_chatbot(user_input):
    
    # Store user message
    store_message("user", user_input)

    # Step 2: Pre-filter
    safety_category = hybrid_prefilter(user_input)

    if safety_category != "SAFE":
        warning = safety_response(safety_category)
        store_message("assistant", warning)
        return warning

    try:
        # Step 3–5: LLM response
        response = get_llm_response(user_input)

        # Safety check
        validation = hybrid_output_validator(response)

        if validation == "UNSAFE":
            response = fallback_response()

    except Exception as e:
        # IMPORTANT: prevents None output
        response = "Sorry, I could not process your request due to a technical error."

    # Store assistant message
    store_message("assistant", response)


    return response

In [21]:
# taking user input and calling main logic function
user_query = input("Ask health question: ")

reply = health_chatbot(user_query)

print("\nBot:", reply)

Ask health question:  i am sick of people



Bot: Please reach out to emergency support or a mental health professional immediately.
